<a href="https://colab.research.google.com/github/RizalFajri95/FaceRecognition/blob/main/deteksi_wajah.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
import shutil, os, glob
import cv2
import numpy as np
import matplotlib.pyplot as plt
from google.colab import files

for folder in ["database", "query"]:
    if os.path.exists(folder):
        shutil.rmtree(folder)
    os.makedirs(folder)

for f in glob.glob("/content/*.jpeg") + glob.glob("/content/*.jpg") + glob.glob("/content/*.png"):
    os.remove(f)
    print(f"Dihapus: {f}")

print("Folder bersih")

print("Library sudah siap")

FACE_CASCADE = cv2.CascadeClassifier(
    cv2.data.haarcascades + "haarcascade_frontalface_default.xml"
)

IMG_SIZE = (100, 100)
VALID_EXT = (".jpg", ".jpeg", ".png")


def list_images(folder):
    """List file gambar valid di folder, sudah terurut.
    Dipakai konsisten di semua tempat supaya tidak ada file
    non-gambar yang ikut terbaca secara tidak sengaja."""
    return sorted(f for f in os.listdir(folder) if f.lower().endswith(VALID_EXT))

uploaded_db = files.upload()
for fname, data in uploaded_db.items():
    with open(f"database/{fname}", "wb") as f:
        f.write(data)
print(f"{len(uploaded_db)} foto disimpan di database")
print("Isi database:", list_images("database"))

uploaded_query = files.upload()
for fname, data in uploaded_query.items():
    with open(f"query/{fname}", "wb") as f:
        f.write(data)
print(f"{len(uploaded_query)} foto disimpan di query")
print("Isi query:", list_images("query"))

def preprocess_face(image_path, show=False):
    img = cv2.imread(image_path)
    if img is None:
        print(f"Gagal membaca: {image_path}")
        return None, None

    gray = cv2.cvtColor(img, cv2.COLOR_BGR2GRAY)
    gray = cv2.equalizeHist(gray)

    faces = FACE_CASCADE.detectMultiScale(
        gray, scaleFactor=1.1, minNeighbors=5, minSize=(30, 30)
    )

    if len(faces) == 0:
        print(f"  Wajah tidak terdeteksi di {os.path.basename(image_path)}, pakai seluruh gambar")
        face_crop = gray
    else:
        x, y, w, h = max(faces, key=lambda f: f[2] * f[3])
        face_crop = gray[y:y + h, x:x + w]

    face_resized = cv2.resize(face_crop, IMG_SIZE)
    face_vector = face_resized.flatten().astype(np.float64)

    if show:
        plt.figure(figsize=(3, 3))
        plt.imshow(face_resized, cmap="gray")
        plt.title(os.path.basename(image_path))
        plt.axis("off")
        plt.show()

    return face_vector, face_resized


print("Fungsi preprocessing siap")

db_vectors, db_labels, db_images = [], [], []

for fname in list_images("database"):
    label = os.path.splitext(fname)[0]
    vec, img = preprocess_face(f"database/{fname}", show=True)
    if vec is not None:
        db_vectors.append(vec)
        db_labels.append(label)
        db_images.append(img)
        print(f"{fname} -> vektor shape: {vec.shape}")

if len(db_vectors) == 0:
    raise RuntimeError(
        "Tidak ada foto valid di folder 'database'. "
        "Pastikan file berekstensi .jpg/.jpeg/.png dan berhasil dibaca."
    )
if len(db_vectors) < 2:
    print("PERINGATAN: database hanya punya 1 foto. PCA butuh beberapa foto "
          "wajah berbeda untuk belajar variasi yang bermakna -- hasil "
          "pengenalan dengan 1 foto kemungkinan besar tidak akurat.")

X = np.array(db_vectors)
print(f"Matriks X: {X.shape}  ({X.shape[0]} wajah x {X.shape[1]} piksel)")

N_COMPONENTS = max(1, min(len(db_vectors) - 1, 50))

mean_face = np.mean(X, axis=0)
X_centered = X - mean_face

U, S, Vt = np.linalg.svd(X_centered, full_matrices=False)

eigenfaces = Vt[:N_COMPONENTS]
eigenvectors = Vt[:N_COMPONENTS].T
X_projected = X_centered @ eigenvectors

print(f"PCA selesai: {N_COMPONENTS} komponen, proyeksi shape: {X_projected.shape}")

n_show = min(6, N_COMPONENTS + 1)
fig, axes = plt.subplots(1, n_show, figsize=(3 * n_show, 3))
if n_show == 1:
    axes = [axes]

axes[0].imshow(mean_face.reshape(IMG_SIZE), cmap="gray")
axes[0].set_title("Mean Face")
axes[0].axis("off")

for i in range(1, n_show):
    ef = eigenfaces[i - 1].reshape(IMG_SIZE)
    rng = ef.max() - ef.min()
    ef = (ef - ef.min()) / rng if rng > 0 else np.zeros_like(ef)
    axes[i].imshow(ef, cmap="gray")
    axes[i].set_title(f"Eigenface {i}")
    axes[i].axis("off")

plt.tight_layout()
plt.show()

variance_ratio = (S[:N_COMPONENTS] ** 2) / np.sum(S ** 2)
cumulative_var = np.cumsum(variance_ratio)

plt.figure(figsize=(7, 3))
plt.plot(range(1, len(cumulative_var) + 1), cumulative_var * 100,
         marker="o", markersize=5, color="#5340d4")
plt.axhline(90, color="coral", linestyle="--", label="90% variansi")
plt.xlabel("Jumlah Komponen PCA")
plt.ylabel("Variansi Kumulatif (%)")
plt.title("Variansi yang Dijelaskan oleh Komponen PCA")
plt.legend()
plt.grid(alpha=0.3)
plt.tight_layout()
plt.show()

print(f"{N_COMPONENTS} komponen menjelaskan {cumulative_var[-1] * 100:.1f}% variansi total")

print("\n=== Kalibrasi Threshold ===")
if len(X_projected) >= 2:
    nearest_other_dists = []
    for i in range(len(X_projected)):
        dists_i = [
            np.linalg.norm(X_projected[i] - X_projected[j])
            for j in range(len(X_projected)) if j != i
        ]
        nearest_other_dists.append(min(dists_i))

    suggested_threshold = float(np.percentile(nearest_other_dists, 25))
    print(f"Rentang jarak antar wajah berbeda di database : "
          f"{min(nearest_other_dists):.1f} - {max(nearest_other_dists):.1f}")
    print(f"Threshold yang disarankan (persentil 25)       : {suggested_threshold:.1f}")
    print("Catatan: ini estimasi awal dari database kamu sendiri. Tetap cek "
          "ulang dengan foto query asli (orang dikenal vs tidak dikenal), "
          "lalu sesuaikan manual kalau perlu.")

    plt.figure(figsize=(6, 3))
    plt.hist(nearest_other_dists, bins=min(10, len(nearest_other_dists)),
             color="#5340d4", alpha=0.7)
    plt.axvline(suggested_threshold, color="coral", linestyle="--",
                label=f"Saran threshold: {suggested_threshold:.0f}")
    plt.xlabel("Jarak terdekat ke wajah lain di database")
    plt.ylabel("Jumlah")
    plt.title("Distribusi jarak antar wajah (bantu menentukan threshold)")
    plt.legend()
    plt.tight_layout()
    plt.show()

    THRESHOLD = round(suggested_threshold)
else:
    print("Database cuma 1 foto, tidak bisa dikalibrasi otomatis. Pakai default.")
    THRESHOLD = 5000

print(f"THRESHOLD yang dipakai: {THRESHOLD}  (ubah manual di bawah kalau hasil kurang pas)")

def recognize_face(query_path, threshold=THRESHOLD):
    print(f"Mengenali: {os.path.basename(query_path)}")

    query_vec, query_img = preprocess_face(query_path, show=True)
    if query_vec is None:
        return None

    query_centered = query_vec - mean_face
    query_projected = query_centered @ eigenvectors

    distances = [
        (np.linalg.norm(query_projected - X_projected[i]), db_labels[i], i)
        for i in range(len(X_projected))
    ]
    distances.sort(key=lambda x: x[0])

    best_dist, best_label, best_idx = distances[0]

    print(f"Jarak terdekat : {best_dist:.2f}")
    print(f"Threshold      : {threshold}")
    if best_dist < threshold:
        print(f"WAJAH DIKENALI sebagai -> {best_label}")
    else:
        print("WAJAH TIDAK DIKENALI (jarak melebihi threshold)")

    fig, axes = plt.subplots(1, 3, figsize=(10, 3.5))

    axes[0].imshow(query_img, cmap="gray")
    axes[0].set_title("Foto Query")
    axes[0].axis("off")

    axes[1].imshow(db_images[best_idx], cmap="gray")
    axes[1].set_title(f"Paling mirip: {best_label}")
    axes[1].axis("off")

    top_n = min(3, len(distances))
    top_labels = [d[1] for d in distances[:top_n]]
    top_dists = [d[0] for d in distances[:top_n]]
    colors = ["#3bb575" if d < threshold else "#e85d30" for d in top_dists]
    axes[2].barh(top_labels[::-1], top_dists[::-1], color=colors[::-1])
    axes[2].axvline(threshold, color="gray", linestyle="--", label=f"Threshold {threshold}")
    axes[2].set_title("Jarak ke database")
    axes[2].set_xlabel("Euclidean Distance")
    axes[2].legend(fontsize=8)

    status = f"DIKENALI -> {best_label}" if best_dist < threshold else "TIDAK DIKENALI"
    plt.suptitle(f"Hasil: {status}", fontsize=13, fontweight="bold")
    plt.tight_layout()
    plt.show()

    return best_label, best_dist


print("Fungsi recognize_face siap")

query_files = list_images("query")
if len(query_files) == 0:
    print("Tidak ada foto valid di folder 'query'.")

results = []
for fname in query_files:
    result = recognize_face(f"query/{fname}", threshold=THRESHOLD)
    if result:
        results.append({"file": fname, "prediksi": result[0], "jarak": result[1]})

print("\nRingkasan Hasil Pengenalan")
for r in results:
    status = "DIKENALI" if r["jarak"] < THRESHOLD else "TIDAK DIKENALI"
    print(f"{r['file']:30s} -> {r['prediksi']:20s} | {status} (dist: {r['jarak']:.0f})")

if query_files:
    query_path = f"query/{query_files[0]}"
    query_vec, _ = preprocess_face(query_path)
    query_centered = query_vec - mean_face

    k_list = sorted(set(k for k in [1, 5, 10, 20, N_COMPONENTS] if k <= N_COMPONENTS))

    fig, axes = plt.subplots(1, len(k_list) + 1, figsize=(3 * (len(k_list) + 1), 3))

    axes[0].imshow(query_vec.reshape(IMG_SIZE), cmap="gray")
    axes[0].set_title("Original")
    axes[0].axis("off")

    for i, k in enumerate(k_list):
        proj_k = query_centered @ Vt[:k].T
        recon = proj_k @ Vt[:k] + mean_face
        recon = np.clip(recon, 0, 255)
        axes[i + 1].imshow(recon.reshape(IMG_SIZE), cmap="gray")
        axes[i + 1].set_title(f"k = {k}")
        axes[i + 1].axis("off")

    plt.suptitle("Rekonstruksi wajah dengan k komponen PCA", fontsize=13)
    plt.tight_layout()
    plt.show()

    print("Semakin besar k = rekonstruksi semakin mendekati gambar asli")
else:
    print("Tidak ada foto query untuk divisualisasikan rekonstruksinya.")